# RAG: Implementation (Production)

In [22]:
from openai import OpenAI
import numpy as np
import faiss
from pypdf import PdfReader
from sentence_transformers import CrossEncoder

In [23]:
from google.colab import userdata
api_key = userdata.get("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

## Load PDF

In [24]:
def load_pdf(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

document = load_pdf("/content/Agreement_for_sale.pdf")

## Chunking

In [25]:
def chunk_text(text, chunk_size=500, overlap=100):
    words = text.split()
    chunks = []

    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap

    return chunks

chunks = chunk_text(document)
print("Total chunks:", len(chunks))

Total chunks: 66


## Embeddings

In [26]:
def embed(texts):
    return [
        client.embeddings.create(
            model="text-embedding-3-small",
            input=t
        ).data[0].embedding
        for t in texts
    ]

embeddings = embed(chunks)

## FAISS Vector DB

In [27]:
dimension = len(embeddings[0])
index = faiss.IndexFlatL2(dimension) # By default, FAISS uses Euclidean distance (L2)

index.add(np.array(embeddings).astype("float32"))

## Retrieval

In [28]:
def retrieve(query, k=10):
    q_emb = embed([query])[0]
    D, I = index.search(np.array([q_emb]).astype("float32"), k)

    return [(chunks[i], D[0][j]) for j, i in enumerate(I[0])]

## Re-ranking (Cross-Encoder)

In [29]:
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query, retrieved):
    chunk_texts = [c for c, _ in retrieved]
    pairs = [[query, c] for c in chunk_texts]

    scores = reranker.predict(pairs)

    reranked = list(zip(chunk_texts, scores))
    return sorted(reranked, key=lambda x: x[1], reverse=True)

## Build Prompt

In [30]:
def answer(query):
    retrieved = retrieve(query, k=10)
    reranked = rerank(query, retrieved)[:3]

    context = "\n".join([c for c, _ in reranked])

    prompt = f"""
Answer using ONLY the context below.

Context:
{context}

Question:
{query}
"""

    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt
    )

    return response.output_text

## RAG in Action

In [31]:
query = "What's the maintenance charges of the society?"

print(answer(query))

The provisional maintenance charge is Rs. 5/- per square foot of (carpet area + adjoining balcony area) per month, charged for 24 months in advance.
